In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("/Users/fuyuxuan/Downloads/testout_1.4.csv")
A = df.to_numpy(dtype=float)

def projection_psd(M: np.ndarray) -> np.ndarray:
    M = (M + M.T) / 2.0
    vals, vecs = np.linalg.eigh(M)
    vals = np.clip(vals, 0.0, None)
    return vecs @ np.diag(vals) @ vecs.T

def higham_near_corr(C: np.ndarray, tol: float = 1e-14, max_iter: int = 10000) -> np.ndarray:
    Y = (A + A.T) / 2.0
    delta_S = np.zeros_like(Y)
    for _ in range(max_iter):
        R = Y - delta_S
        X = projection_psd(R)
        delta_S = X - R

        Y_new = X.copy()
        np.fill_diagonal(Y_new, 1.0)

        if np.linalg.norm(Y_new - Y, ord="fro") < tol:
            Y = Y_new
            break
        Y = Y_new
        
    return (Y + Y.T) / 2.0

C_high = higham_near_corr(A)

out = pd.DataFrame(C_high, columns=df.columns)
print(out)


         x1        x2        x3        x4        x5
0  1.000000 -0.483199 -0.241787 -0.067767 -0.714761
1 -0.483199  1.000000  0.015446  0.405660  0.178286
2 -0.241787  0.015446  1.000000  0.488250  0.336248
3 -0.067767  0.405660  0.488250  1.000000 -0.322136
4 -0.714761  0.178286  0.336248 -0.322136  1.000000
